# Two-Stage Ridge: Public Research Demonstration

A pool-level Ridge model supplies a shared component; a second Ridge model learns each target's residual. The public example uses fold-local imputation and standardization. Its independent synthetic features and small grids are not the competition recipe.

**Scope:** this runnable notebook demonstrates the workflow, not the reported competition score. No competition data or private factors are loaded. See [research limitations](../docs/RESEARCH.md).


## 1. Reproducible synthetic panel

The public generator is unrelated to the withheld feature construction. All three model notebooks use the same ordered row IDs and day groups.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "src" / "quant_portfolio").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
ARTIFACT_DIR = Path(os.environ.get("PORTFOLIO_ARTIFACT_DIR", str(ROOT / "artifacts")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from quant_portfolio.data import make_synthetic_panel
from quant_portfolio.blend import weighted_directional_accuracy
from quant_portfolio.models import model_oof, refit_predict
from quant_portfolio.artifacts import save_model_artifacts

panel = make_synthetic_panel(seed=42)
print("Independent synthetic data only:", panel.train.shape, panel.test.shape)


In [ ]:
display(panel.train.head())
print("Training days:", panel.train.day_id.nunique())
print("Missing synthetic covariates:", int(panel.train.filter(like="feature_").isna().sum().sum()))


## 2. Hyperparameter selection

Selection uses five day-grouped folds. The resulting best OOF score is a selection diagnostic, not an independent estimate. The first run is intentionally small for review on a laptop.

The coarse search is followed by a local two-dimensional grid with an absolute step of 0.01. This is a finite search, not a guarantee of a global optimum.


In [ ]:
from quant_portfolio.search import ridge_grid_search
selected = ridge_grid_search(panel)
params = selected.params
display(selected.table)
print("Chosen demonstration parameters:", params)


## 3. Rebuild aligned OOF predictions

Each evaluation day is absent from that fit's training rows. The feature provider receives training labels only. The same parameters are then used for the full-data refit. Both model calls attach a generation context to their predictions, including the data, implementation, provider, parameters, seed, runtime, and OOF fold protocol.


In [ ]:
MODEL = "ridge"
oof = model_oof(MODEL, panel, params, n_splits=5, seed=42)
score = weighted_directional_accuracy(panel.y, oof)
print(f"Synthetic OOF selection score: {score:.6f}")
assert abs(score - selected.score) <= 1e-12
assert oof.index.equals(panel.train.index)


## 4. Full-data fit and prediction artifacts

Fit on all training rows and predict the disjoint unlabeled synthetic test panel. `save_model_artifacts` accepts only a matching OOF/full-refit pair with the context attached by the model functions. It writes an immutable generation under `ridge/runs/<run_id>/`, verifies the payloads, and atomically switches `ridge/manifest.json` only after the generation is complete. Earlier runs remain available. These are not competition submission files.


In [ ]:
test_raw = refit_predict(MODEL, panel, params, seed=42)
manifest = save_model_artifacts(ARTIFACT_DIR, MODEL, panel, oof, test_raw, params)
display(pd.DataFrame(manifest["files"]).T)
display(test_raw.head())


## 5. Numerical engineering: one decomposition, many penalties

The private research used spectral reuse for repeated Ridge fits. This public solver exposes the general linear-algebra method using a fresh artificial matrix, without defining any competition features. Weighted centering leaves the intercept unpenalized; optional scaling uses training statistics. One decomposition supports every alpha below. This section is a numerical equivalence check, not a prediction-performance claim, and it does not replace the two-stage training loop above.


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from quant_portfolio.ridge_path import WeightedRidgePath

rng = np.random.default_rng(7)
design = rng.normal(size=(40, 6))
response = 2.0 + design @ rng.normal(size=6) + rng.normal(size=40)
evaluation = rng.normal(size=(8, 6))
training_weights = np.linspace(0.5, 2.0, len(design))
alphas = np.logspace(-3, 3, 15)

path = WeightedRidgePath(standardize=True).fit(design, response, training_weights)
predictions = path.predict(evaluation, alphas)
scaler = StandardScaler().fit(design, sample_weight=training_weights)
reference = np.column_stack([
    Ridge(alpha=float(alpha), solver="svd")
    .fit(scaler.transform(design), response, sample_weight=training_weights)
    .predict(scaler.transform(evaluation))
    for alpha in alphas
])
np.testing.assert_allclose(predictions, reference, rtol=1e-9, atol=1e-9)
print("Maximum absolute difference from independent Ridge fits:", float(np.abs(predictions - reference).max()))
print("One training decomposition supports", len(alphas), "penalties.")


## Interpretation

There is no held-out synthetic test score: test outcomes are not exposed by the public data API. Model-search results reuse OOF labels, so further independent evaluation would be needed for a generalization claim. Continue to the ensemble notebook only after all three model artifacts have been generated in the same output directory. Legacy schema-1 artifacts are intentionally not upgraded in place; select a fresh output directory and rerun notebooks 01–03.
